# Wikipedia Data Collection Renewable Energy Topics

Notebook ini mengumpulkan artikel Wikipedia berbahasa Indonesia terkait topik "Energi Terbarukan", menyimpannya ke dalam file CSV, lalu melakukan analisis dasar (jumlah kata per artikel dan kata yang paling sering muncul) sebagai data persiapan untuk project NLP lanjutan.

## 1. Instalasi Library

In [1]:
pip install wikipedia-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 651.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 5.8 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.2
    Uninstalling click-8.4.2:
      Successfully uninstalled click-8.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.23.0 requires click<9.0.0,>=8.4.2, but you have click 8.4.1 which is incompatible.


In [2]:
pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=b07c8e3924c6cc10fe006e6452662fa1f7db2fb8b1ba15063c12242517d40c46
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


## 2. Import dan Konfigurasi

In [3]:
import wikipedia
import csv
from collections import Counter
import re

wikipedia.set_lang("id")

TOPIC = "Energi Terbarukan"

print(f"Topik yang dipilih: {TOPIC}")

Topik yang dipilih: Energi Terbarukan


## 3. Fungsi Pengambilan Artikel

Mencari artikel Wikipedia berdasarkan topik, dengan penanganan khusus untuk halaman disambiguasi (memilih opsi pertama) dan halaman yang tidak ditemukan (dilewati).

In [4]:
def get_articles(topic, num_articles=10):
    """
    Mengambil artikel dari Wikipedia berdasarkan topik
    """
    print(f"Mencari artikel tentang: {topic}")

    search_results = wikipedia.search(topic, results=num_articles * 2)

    articles = []

    for title in search_results:
        if len(articles) >= num_articles:
            break

        try:
            print(f"Mengambil artikel: {title}")
            page = wikipedia.page(title, auto_suggest=False)

            articles.append({
                'title': page.title,
                'url': page.url,
                'content': page.content,
                'summary': page.summary
            })

        except wikipedia.exceptions.DisambiguationError as e:
            print(f"  Disambiguasi: {title} - mengambil pilihan pertama")
            try:
                page = wikipedia.page(e.options[0], auto_suggest=False)
                articles.append({
                    'title': page.title,
                    'url': page.url,
                    'content': page.content,
                    'summary': page.summary
                })
            except:
                continue

        except wikipedia.exceptions.PageError:
            print(f"  Halaman tidak ditemukan: {title}")
            continue

        except Exception as e:
            print(f"  Error pada {title}: {str(e)}")
            continue

    return articles

## 4. Fungsi Penyimpanan ke CSV

Setiap artikel disimpan dengan delimiter titik koma (`;`) dan seluruh field dibungkus tanda kutip, agar konten artikel yang panjang dan mengandung koma tidak merusak struktur kolom CSV.

In [5]:
def save_to_csv(articles, filename='artikel_energi_terbarukan.csv'):
    """
    Menyimpan hasil ke file CSV dengan konten lengkap dan format rapi
    Setiap kolom dipisahkan dengan delimiter yang jelas
    """
    print(f"\nMenyimpan hasil ke {filename}...")

    with open(filename, 'w', newline='', encoding='utf-8-sig') as file:
        writer = csv.writer(file, delimiter=';', quoting=csv.QUOTE_ALL,
                          quotechar='"', doublequote=True)

        writer.writerow(['No', 'Judul', 'URL', 'Konten', 'Ringkasan'])

        for idx, article in enumerate(articles, 1):
            clean_content = article['content'].replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
            clean_summary = article['summary'].replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

            clean_content = clean_content.replace('"', "'")
            clean_summary = clean_summary.replace('"', "'")

            clean_content = ' '.join(clean_content.split())
            clean_summary = ' '.join(clean_summary.split())

            clean_content = clean_content.replace(';', ',')
            clean_summary = clean_summary.replace(';', ',')

            if len(clean_content) > 32000:
                clean_content = clean_content[:32000] + "..."

            writer.writerow([
                str(idx),
                article['title'],
                article['url'],
                clean_content,
                clean_summary
            ])

    print(f"Berhasil menyimpan {len(articles)} artikel ke {filename}")
    print(f"Format: Setiap kolom dipisahkan dengan ';' dan semua field dibungkus tanda kutip")
    return filename

## 5. Fungsi Analisis: Jumlah Kata per Artikel

Fungsi ini membaca kembali file CSV yang sudah disimpan, sebagai verifikasi bahwa data yang diekspor dapat dibaca ulang dengan benar sebelum dianalisis lebih lanjut.

In [6]:
def count_words_from_csv(filename='artikel_energi_terbarukan.csv'):
    """
    Membaca file CSV dan menghitung jumlah kata per artikel
    """
    print("\n" + "=" * 60)
    print("MENGHITUNG JUMLAH KATA PER ARTIKEL DARI CSV")
    print("=" * 60)

    articles_stats = []

    try:
        with open(filename, 'r', encoding='utf-8-sig') as file:
            reader = csv.DictReader(file, delimiter=';')

            fieldnames = reader.fieldnames
            print(f"Kolom yang tersedia: {fieldnames}\n")

            for row in reader:
                content = row['Konten']
                words = re.findall(r'\b\w+\b', content.lower())
                word_count = len(words)

                articles_stats.append({
                    'no': row['No'],
                    'title': row['Judul'],
                    'url': row['URL'],
                    'word_count': word_count,
                    'content': content
                })

                print(f"{row['No']}. {row['Judul']}")
                print(f"   Jumlah kata: {word_count:,}")
                print(f"   URL: {row['URL']}\n")

        return articles_stats

    except KeyError as e:
        print(f"Error: Kolom {e} tidak ditemukan dalam CSV")
        print("Pastikan file CSV memiliki kolom: No, Judul, URL, Konten, Ringkasan")
        return []
    except FileNotFoundError:
        print(f"Error: File {filename} tidak ditemukan")
        return []
    except Exception as e:
        print(f"Error tidak terduga: {str(e)}")
        return []

## 6. Fungsi Analisis: Kata Paling Sering Muncul

Menghitung frekuensi kemunculan kata di seluruh artikel (setelah membuang stopword Bahasa Indonesia) untuk melihat topik/istilah yang paling dominan dibahas.

In [7]:
def get_most_common_words_from_csv(filename='artikel_energi_terbarukan.csv', top_n=10):
    """
    Membaca file CSV dan mengidentifikasi kata paling sering muncul
    """
    print("=" * 60)
    print("IDENTIFIKASI 10 KATA PALING SERING MUNCUL DARI CSV")
    print("=" * 60)

    all_words = []

    stopwords = set([
        'yang', 'dan', 'di', 'dari', 'untuk', 'pada', 'dengan', 'ini', 'itu',
        'adalah', 'ke', 'dalam', 'oleh', 'sebagai', 'akan', 'atau', 'juga',
        'telah', 'dapat', 'lebih', 'seperti', 'tidak', 'ada', 'tersebut',
        'karena', 'antara', 'sama', 'bisa', 'namun', 'hingga', 'saat', 'bila',
        'sudah', 'hanya', 'masih', 'saja', 'sebuah', 'suatu', 'mereka', 'kita'
    ])

    try:
        with open(filename, 'r', encoding='utf-8-sig') as file:
            reader = csv.DictReader(file, delimiter=';')

            for row in reader:
                content = row['Konten']
                words = re.findall(r'\b\w+\b', content.lower())

                filtered_words = [w for w in words if len(w) > 3 and w not in stopwords]
                all_words.extend(filtered_words)

        word_counts = Counter(all_words)
        most_common = word_counts.most_common(top_n)

        for idx, (word, count) in enumerate(most_common, 1):
            print(f"{idx}. {word}: {count} kali")

        return most_common

    except KeyError as e:
        print(f"Error: Kolom {e} tidak ditemukan dalam CSV")
        return []
    except FileNotFoundError:
        print(f"Error: File {filename} tidak ditemukan")
        return []
    except Exception as e:
        print(f"Error tidak terduga: {str(e)}")
        return []

## 7. Eksekusi Program

In [8]:
def main():
    print("=" * 60)
    print("PROGRAM PENGAMBILAN ARTIKEL WIKIPEDIA - ENERGI TERBARUKAN")
    print("=" * 60)

    print(f"\nTopik dipilih = '{TOPIC}'")

    print("\n" + "=" * 60)
    print("MENGAMBIL 10 ARTIKEL DARI WIKIPEDIA")
    print("=" * 60)
    articles = get_articles(TOPIC, num_articles=10)
    print(f"\nBerhasil mengambil {len(articles)} artikel")

    print("\n" + "=" * 60)
    print("MENYIMPAN HASIL KE CSV")
    print("=" * 60)
    csv_filename = save_to_csv(articles)

    articles_stats = count_words_from_csv(csv_filename)

    most_common = get_most_common_words_from_csv(csv_filename, top_n=10)

    print("\n" + "=" * 60)
    print("PROGRAM SELESAI!")
    print("=" * 60)
    print(f"File CSV tersimpan di: {csv_filename}")
    print(f"Total artikel: {len(articles_stats)}")

if __name__ == "__main__":
    main()

PROGRAM PENGAMBILAN ARTIKEL WIKIPEDIA - ENERGI TERBARUKAN

Topik dipilih = 'Energi Terbarukan'

MENGAMBIL 10 ARTIKEL DARI WIKIPEDIA
Mencari artikel tentang: Energi Terbarukan
Mengambil artikel: Kementerian Energi dan Sumber Daya Mineral Republik Indonesia
Mengambil artikel: Energi terbarukan
Mengambil artikel: Energi terbarukan di Skotlandia
Mengambil artikel: Energi terbarukan di Indonesia
Mengambil artikel: Energi tak terbarukan
Mengambil artikel: Direktorat Jenderal Energi Baru, Terbarukan, dan Konservasi Energi
Mengambil artikel: Energi
Mengambil artikel: Pertamina
Mengambil artikel: Energy Management Indonesia
Mengambil artikel: Energi Mega Persada

Berhasil mengambil 10 artikel

MENYIMPAN HASIL KE CSV

Menyimpan hasil ke artikel_energi_terbarukan.csv...
Berhasil menyimpan 10 artikel ke artikel_energi_terbarukan.csv
Format: Setiap kolom dipisahkan dengan ';' dan semua field dibungkus tanda kutip

MENGHITUNG JUMLAH KATA PER ARTIKEL DARI CSV
Kolom yang tersedia: ['No', 'Judul', 'URL